# 01 – Veri Hazırlama
Bu notebook İngilizce (Twitter US Airline Sentiment) ve Türkçe (winvoker) veri setlerini yükler, etiketleri ortak biçime çevirir, temizler ve tüm yöntemlerin kullanacağı test ve eğitim dosyalarını üretir.

**Çıktılar:** `data/test_en.csv`, `data/test_tr.csv`, `data/train_tr.csv` (UTF-8, sütunlar `id, text, label`)
**Ortam:** Google Colab; veri Drive'daki `DuyguAnalizi` klasöründe.

In [3]:
from google.colab import drive
drive.mount('/content/drive')

KLASOR = '/content/drive/MyDrive/DuyguAnalizi'


Mounted at /content/drive


## 1. Drive bağlantısı
Ham veri ve çıktılar Google Drive'daki `DuyguAnalizi` klasöründe tutulur. `ham_veri/Tweets.csv` dosyası Kaggle'dan indirilip bu klasöre konmuştur.

In [4]:
import pandas as pd

en = pd.read_csv(f'{KLASOR}/ham_veri/Tweets.csv')
print(en.shape)
print(en['airline_sentiment'].value_counts())

(14640, 15)
airline_sentiment
negative    9178
neutral     3099
positive    2363
Name: count, dtype: int64


## 2. İngilizce veri: Twitter US Airline Sentiment
Kaggle'daki veri seti: ABD'deki altı havayolu hakkında 14.640 tweet. Etiketler: negative / neutral / positive.

In [5]:
en = en[['text', 'airline_sentiment']]
en = en.rename(columns={'airline_sentiment': 'label'})

etiket_en = {'positive': 'Pozitif', 'negative': 'Negatif', 'neutral': 'Nötr'}
en['label'] = en['label'].map(etiket_en)

print(en.head())
print(en['label'].value_counts())


                                                text    label
0                @VirginAmerica What @dhepburn said.     Nötr
1  @VirginAmerica plus you've added commercials t...  Pozitif
2  @VirginAmerica I didn't today... Must mean I n...     Nötr
3  @VirginAmerica it's really aggressive to blast...  Negatif
4  @VirginAmerica and it's a really big bad thing...  Negatif
label
Negatif    9178
Nötr       3099
Pozitif    2363
Name: count, dtype: int64


### 2.1 Sütun seçimi ve etiket dönüşümü
Yalnızca metin (`text`) ve duygu (`airline_sentiment`) sütunları tutulur. Etiketler ortak biçime çevrilir: positive → Pozitif, negative → Negatif, neutral → Nötr.

In [6]:
print('Önce:', len(en))

en = en.dropna(subset=['text', 'label'])
en = en.drop_duplicates(subset=['text'])
en = en.reset_index(drop=True)

print('Sonra:', len(en))
print(en['label'].value_counts())

Önce: 14640
Sonra: 14427
label
Negatif    9080
Nötr       3057
Pozitif    2290
Name: count, dtype: int64


### 2.2 Temizlik
Boş satırlar ve metni birebir aynı olan tekrar eden satırlar silinir, böylece aynı cümle hem eğitime hem teste düşmez. Metin küçük harfe çevrilmez, noktalama ve emojiler silinmez, çünkü VADER bunları duygu işareti olarak kullanır.

In [7]:
from datasets import load_dataset

tr_ds = load_dataset('winvoker/turkish-sentiment-analysis-dataset')
print(tr_ds)

README.md:   0%|          | 0.00/1.99k [00:00<?, ?B/s]

train.csv: reconstructing file:   0%|          |  0.00B / 76.1MB            

train.csv: downloading bytes:           |  0.00B            

test.csv:   0%|          | 0.00/8.47M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/440679 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/48965 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['text', 'label', 'dataset'],
        num_rows: 440679
    })
    test: Dataset({
        features: ['text', 'label', 'dataset'],
        num_rows: 48965
    })
})


## 3. Türkçe veri: winvoker Turkish Sentiment Analysis Dataset
Hugging Face'ten doğrudan indirilir. Veri seti eğitim (440.679) ve test (48.965) bölümleriyle gelir; sütunlar `text`, `label`, `dataset`.

In [8]:
tr = pd.concat([tr_ds['train'].to_pandas(), tr_ds['test'].to_pandas()], ignore_index=True)

print(tr.shape)
print(tr['label'].value_counts())
print(tr['dataset'].value_counts())

(489644, 3)
label
Positive    262166
Notr        170917
Negative     56561
Name: count, dtype: int64
dataset
urun_yorumlari      234183
wiki                170413
HUMIR                65000
tweet-pn             11055
magaza_yorumlari      8489
random                 504
Name: count, dtype: int64


### 3.2 Etiket dönüşümü ve temizlik
İngilizce veriyle aynı adımlar uygulanır: Positive → Pozitif, Negative → Negatif, Notr → Nötr; boş ve tekrar eden satırlar silinir.

In [9]:
tr = tr[['text', 'label']]

etiket_tr = {'Positive': 'Pozitif', 'Negative': 'Negatif', 'Notr': 'Nötr'}
tr['label'] = tr['label'].map(etiket_tr)

print('Önce:', len(tr))
tr = tr.dropna(subset=['text', 'label'])
tr = tr.drop_duplicates(subset=['text'])
tr = tr.reset_index(drop=True)
print('Sonra:', len(tr))

print(tr['label'].value_counts())

Önce: 489644
Sonra: 488879
label
Pozitif    261688
Nötr       170916
Negatif     56275
Name: count, dtype: int64


## 4. Test kümeleri
Sınıf dengesizliğinin sonuçları etkilememesi için her sınıftan rastgele **1.000** örnek seçilir (`random_state=42`). Satırlar karıştırılır ve her birine benzersiz bir `id` verilir. Üç yöntem de yalnızca bu dosyalarda değerlendirilir.

### 4.1 İngilizce test kümesi (`test_en`)

In [10]:
test_en = en.groupby('label').sample(n=1000, random_state=42)
test_en = test_en.sample(frac=1, random_state=42).reset_index(drop=True)
test_en.insert(0, 'id', ['en_' + str(i) for i in range(len(test_en))])

print(test_en.shape)
print(test_en['label'].value_counts())
print(test_en.head())

(3000, 3)
label
Nötr       1000
Negatif    1000
Pozitif    1000
Name: count, dtype: int64
     id                                               text    label
0  en_0  @SouthwestAir's CEO Kelly draws record crowd t...     Nötr
1  en_1  @SouthwestAir How do I retroactively add previ...     Nötr
2  en_2  @JetBlue woof I'm on the red cArpet http://t.c...     Nötr
3  en_3  @USAirways That's 1h 10m waiting to get my bag...  Negatif
4  en_4  @united thank you! My second flight I already ...  Pozitif


### 4.2 Türkçe test kümesi (`test_tr`)

In [11]:
test_tr = tr.groupby('label').sample(n=1000, random_state=42)
test_tr = test_tr.sample(frac=1, random_state=42).reset_index(drop=True)
test_tr.insert(0, 'id', ['tr_' + str(i) for i in range(len(test_tr))])

print(test_tr.shape)
print(test_tr['label'].value_counts())
print(test_tr.head())

(3000, 3)
label
Nötr       1000
Negatif    1000
Pozitif    1000
Name: count, dtype: int64
     id                                               text    label
0  tr_0                Alman kıyı savunması da ateş açtı .     Nötr
1  tr_1               Faaliyetler gençler düzeyinde oldu .     Nötr
2  tr_2  O zamanki yönetimlerin özel sektöre getirdiği ...     Nötr
3  tr_3  saldiriya ugradik berbat bir oteldi odalarda m...  Negatif
4  tr_4       ekonomik hızlı ve orijinal daha ne olsun :))  Pozitif


## 5. Türkçe eğitim kümesi (`train_tr`)
Yalnızca BERTurk ince ayarı için kullanılır. Test kümesindeki cümleler çıkarıldıktan sonra kalan veriden her sınıf için 10.000 örnek seçilir. Son satırdaki kontrol, eğitim ve test kümelerinde ortak cümle olmadığını doğrular (sonuç 0 olmalı).

In [12]:
kalan_tr = tr[~tr['text'].isin(test_tr['text'])]

train_tr = kalan_tr.groupby('label').sample(n=10000, random_state=42)
train_tr = train_tr.sample(frac=1, random_state=42).reset_index(drop=True)
train_tr.insert(0, 'id', ['trn_' + str(i) for i in range(len(train_tr))])

print(len(tr), len(kalan_tr))
print(train_tr['label'].value_counts())
print('Ortak satır:', train_tr['text'].isin(test_tr['text']).sum())

488879 485879
label
Negatif    10000
Pozitif    10000
Nötr       10000
Name: count, dtype: int64
Ortak satır: 0


## 6. Dosyaları kaydetme
Üç dosya Drive'daki `data/` klasörüne UTF-8 CSV olarak yazılır. Aynı dosyalar repodaki `data/` klasöründe de bulunur.

In [13]:
import os

os.makedirs(f'{KLASOR}/data', exist_ok=True)

test_en.to_csv(f'{KLASOR}/data/test_en.csv', index=False, encoding='utf-8')
test_tr.to_csv(f'{KLASOR}/data/test_tr.csv', index=False, encoding='utf-8')
train_tr.to_csv(f'{KLASOR}/data/train_tr.csv', index=False, encoding='utf-8')

print(os.listdir(f'{KLASOR}/data'))

['test_en.csv', 'test_tr.csv', 'train_tr.csv']
